In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger("PipelineDBUCostReporter")

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("overlap_days", "3", "Overlap days (min 2)")
dbutils.widgets.text("workspace_ids", "", "Workspace IDs (comma-separated, blank=all)")

In [ ]:
# =======================================================
# Pipeline DBU Cost Client
# =======================================================
# Sibling of PoolDBUCostClient. Collects declarative / serverless
# pipeline-backed DBU into the dbspend360_pipeline_dbu_cost staging table.
# Differences vs the pool collector:
#   * usage filter:          usage_metadata.dlt_pipeline_id IS NOT NULL
#                            (the canonical declarative-pipeline signal -
#                            captures DLT, DBSQL materialized views /
#                            streaming tables, online tables, vector search,
#                            model serving, AI functions; BOTH serverless and
#                            classic). See plan §3.1.
#   * price join:            INNER (not LEFT) + two-directional PRICE_JOIN_*
#                            guard. A DROP means a SKU lost its list price
#                            (silent undercount); a FAN_OUT means a SKU matched
#                            >1 overlapping price row (silent cost inflation).
#                            Assert the join is exactly 1:1. See plan §5.4.
#   * aggregation key:       (workspace_id, pipeline_id, usage_date,
#                            cluster_id, billing_origin_product). workspace_id
#                            is in the key because pipeline_id is only unique
#                            within a workspace. cluster_id is kept (NOT
#                            coalesced to a sentinel): NULL is the serverless
#                            signal and the v2 cloud-cost join is cluster_id-
#                            keyed. billing_origin_product stays in the grain
#                            so the per-workload $ split is exact downstream.
#   * compute_mode (per group): serverless when the (cluster, product) group
#                            has cluster_id IS NULL, OR a serverless-only
#                            billing_origin_product (MODEL_SERVING /
#                            VECTOR_SEARCH / AI_FUNCTIONS), OR any SKU whose
#                            name carries SERVERLESS; else classic. Keying ONLY
#                            on cluster_id IS NULL mislabeled serverless Model
#                            Serving / AI Functions endpoints (they carry a
#                            non-null "-v2n" cluster_id but run in Databricks'
#                            account -> no customer EC2) as classic. Derived as
#                            an AGGREGATE so it stays a function of the MERGE
#                            key and never splits one key into two source rows.
#   * update/maintenance:    split into update_cost / maintenance_cost from
#                            dlt_update_id / dlt_maintenance_id (cheap, both
#                            ids already read). Their sum need NOT equal
#                            databricks_cost (a row can carry neither sub-id).
#   * MERGE key:             (workspace_id, pipeline_id, usage_date,
#                            cluster_id, billing_origin_product). cluster_id is
#                            matched with NULL-SAFE equality (<=> / eqNullSafe)
#                            because serverless rows carry cluster_id = NULL; a
#                            plain '=' never matches NULL and would re-insert
#                            ~96% of the spend on every overlapping run. See
#                            plan §3.3 / §5.4.
class PipelineDBUCostClient:

    TABLE_NAME = "dbspend360_pipeline_dbu_cost"

    def __init__(self, audit_table: str, target_table: str, covered_table: str, overlap_days: int, logger=None):
        self.audit_table = audit_table
        self.target_table = target_table
        self.covered_table = covered_table
        self.overlap_days = overlap_days
        self.logger = logger or logging.getLogger("PipelineDBUCostClient")
        raw_ws = dbutils.widgets.get("workspace_ids")
        if raw_ws.strip() == "":
            self.workspace_ids = None
        else:
            self.workspace_ids = [w.strip() for w in raw_ws.split(",") if w.strip()]

    def compute_and_merge_dbu_cost(self):
        start_dt = end_dt = datetime.now(timezone.utc).date()
        try:
            start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

            valid, msg = validate_date_window(start_dt, end_dt)
            if not valid:
                raise DataQualityError(msg)

            self.logger.info(f"Loading pipeline DBU cost from {start_dt} to {end_dt}")

            # Primary filter: any billing row tagged with a dlt_pipeline_id.
            # Per plan §3.1 this captures BOTH serverless (cluster_id NULL) and
            # classic, across ALL declarative products. billing_origin_product
            # is carried for dimensioning/labelling - never used to drop rows.
            # Pre-project the usage side to clean, uniquely-named columns
            # BEFORE the join (all struct access resolved here, no qualified
            # refs survive into the join). u_sku_name is renamed so it never
            # collides with the price table's sku_name. On serverless Spark
            # Connect, a column that participates in an equi-join key becomes
            # unresolvable when referenced (qualified) AFTER the join - even
            # with no name collision - so we carry sku through under u_sku_name
            # and never touch a join-key column post-join. See plan §5.4.
            usage_df = (
                spark.table("system.billing.usage")
                     .filter(
                         (F.col("usage_date") >= F.lit(start_dt)) &
                         (F.col("usage_date") <= F.lit(end_dt)) &
                         (F.col("usage_metadata")["dlt_pipeline_id"].isNotNull())
                     )
            )
            if self.workspace_ids is not None:
                usage_df = usage_df.filter(F.col("workspace_id").isin(self.workspace_ids))

            u = usage_df.select(
                F.col("workspace_id"),
                F.col("usage_metadata")["dlt_pipeline_id"].alias("pipeline_id"),
                F.col("usage_date"),
                # cluster_id kept (NULL for serverless) - NOT a sentinel - so
                # the v2 cloud-cost join needs no re-ingest.
                F.col("usage_metadata")["cluster_id"].alias("cluster_id"),
                F.col("billing_origin_product"),
                F.col("sku_name").alias("u_sku_name"),
                F.col("usage_start_time"),
                F.col("usage_quantity"),
                F.col("usage_metadata")["dlt_update_id"].alias("dlt_update_id"),
                F.col("usage_metadata")["dlt_maintenance_id"].alias("dlt_maintenance_id"),
            )

            lp = spark.table("system.billing.list_prices").select(
                F.col("sku_name").alias("lp_sku_name"),
                F.col("price_start_time"),
                F.col("price_end_time"),
                F.col("pricing"),
            )

            # INNER join (council fix): a missing/renamed SKU must NOT silently
            # vanish (LEFT join + NULL price would produce a NULL row_cost).
            # Every column now has a unique name, so the condition and all
            # downstream references are unambiguous bare names.
            joined = u.join(
                lp,
                on=(
                    (F.col("u_sku_name") == F.col("lp_sku_name")) &
                    (F.col("usage_start_time") >= F.col("price_start_time")) &
                    (
                        (F.col("usage_start_time") < F.col("price_end_time")) |
                        F.col("price_end_time").isNull()
                    )
                ),
                how="inner",
            )

            # Two-directional 1:1 guard (plan §5.4). A DROP (join < left) means a
            # SKU had no list price and its usage silently vanished (SKU drift).
            # A FAN_OUT (join > left) means a SKU matched >1 overlapping price
            # row and its cost is silently MULTIPLIED. Checking only the drop
            # direction would miss the inflation case. Assert equality.
            #
            # Cost note (deliberate): these two .count() actions add two scans of
            # the filtered usage set purely for the 1:1 assertion; correctness
            # over speed is the intended trade-off. Cache the projected usage
            # set first.
            u = safe_cache(u)
            left_cnt = u.count()
            join_cnt = joined.count()
            if join_cnt != left_cnt:
                direction = "DROP" if join_cnt < left_cnt else "FAN_OUT"
                self.logger.warning(
                    "PRICE_JOIN_%s: usage rows %d -> joined rows %d (delta %+d). "
                    "Expected 1:1 (one list price per SKU/time). Investigate "
                    "before trusting totals.",
                    direction, left_cnt, join_cnt, join_cnt - left_cnt,
                )
                raise DataQualityError(
                    f"PRICE_JOIN_{direction}: usage rows {left_cnt} -> "
                    f"joined rows {join_cnt} (delta {join_cnt - left_cnt:+d}); "
                    "expected exactly one list-price row per usage row"
                )

            # All columns are already clean, uniquely-named, and struct-free.
            base = joined.select(
                F.col("workspace_id"),
                F.col("pipeline_id"),
                F.col("usage_date"),
                F.col("cluster_id"),
                F.col("billing_origin_product"),
                F.col("u_sku_name").alias("sku_name"),
                F.col("usage_quantity"),
                F.col("pricing")["default"].cast("double").alias("price"),
                F.col("dlt_update_id"),
                F.col("dlt_maintenance_id"),
            )

            # Row-level serverless signal. The original heuristic keyed ONLY on
            # cluster_id IS NULL, which mislabeled serverless Model Serving /
            # Vector Search / AI Functions endpoints (they carry a NON-null
            # "-v2n"-style cluster_id on serverless SKUs) as classic. Those
            # products run in Databricks' own account -> there is NO customer
            # EC2 line, so they must be 'serverless'. Three independent signals,
            # OR'd: cluster_id IS NULL (serverless DLT / MV / ...); a serverless-
            # only billing_origin_product; or an SKU whose name carries
            # SERVERLESS (catch-all for serverless SKUs sitting on a non-null id).
            serverless_only_products = ["MODEL_SERVING", "VECTOR_SEARCH", "AI_FUNCTIONS"]

            with_cols = (
                base
                .withColumn(
                    "is_serverless_row",
                    (
                        F.col("cluster_id").isNull()
                        | F.col("billing_origin_product").isin(serverless_only_products)
                        | F.upper(F.col("sku_name")).like("%SERVERLESS%")
                    ),
                )
                .withColumn("row_cost", F.col("usage_quantity") * F.col("price"))
                .withColumn("is_update", F.col("dlt_update_id").isNotNull())
                .withColumn("is_maint", F.col("dlt_maintenance_id").isNotNull())
            )

            agg_df = (
                with_cols
                .groupBy(
                    "workspace_id",
                    "pipeline_id",
                    "usage_date",
                    "cluster_id",
                    "billing_origin_product",
                )
                .agg(
                    F.sum("row_cost").alias("databricks_cost"),
                    F.sum(F.when(F.col("is_update"), F.col("row_cost"))).alias("update_cost"),
                    F.sum(F.when(F.col("is_maint"), F.col("row_cost"))).alias("maintenance_cost"),
                    F.concat_ws(
                        " + ",
                        F.array_sort(F.collect_set("sku_name")),
                    ).alias("sku_name"),
                    # compute_mode is derived as an AGGREGATE (not a group key)
                    # so it stays a deterministic function of the MERGE key
                    # (cluster_id, billing_origin_product): if ANY row in the
                    # group is serverless the whole (cluster, product) group is
                    # serverless. Keeping compute_mode in the groupBy could split
                    # one MERGE key into two source rows when the SKU signal
                    # disagrees within a group -> an ambiguous (multi-source)
                    # MERGE match. bool aggregation via max-of-int is Spark-
                    # version-agnostic (bool_or is newer).
                    F.max(F.col("is_serverless_row").cast("int")).alias("is_serverless_grp"),
                )
                .withColumn(
                    "compute_mode",
                    F.when(F.col("is_serverless_grp") > 0, F.lit("serverless"))
                     .otherwise(F.lit("classic")),
                )
                .drop("is_serverless_grp")
                .withColumn("currency", F.lit("USD"))
            )

            agg_df = add_workspace_covered(agg_df, self.covered_table, "workspace_id")

            dbu_inc_df = agg_df.select(
                "workspace_id",
                "pipeline_id",
                "usage_date",
                "cluster_id",
                "billing_origin_product",
                "compute_mode",
                "databricks_cost",
                "update_cost",
                "maintenance_cost",
                "currency",
                "sku_name",
                "workspace_covered",
            )

            if dbu_inc_df.limit(1).count() == 0:
                # Empty window is a legitimate state (no declarative-pipeline
                # usage in the period). Don't fail - log an explanatory INFO and
                # write a SUCCESS audit row with row_count=0 below.
                self.logger.info(
                    "No pipeline DBU rows after filtering / aggregation. "
                    "Verify there exists system.billing.usage data with "
                    "usage_metadata.dlt_pipeline_id IS NOT NULL in the window."
                )
                merged_row_count = 0
            else:
                dbu_inc_df = (
                    dbu_inc_df
                    .withColumn("created_at", F.current_timestamp())
                    .withColumn("updated_at", F.current_timestamp())
                )
                dbu_inc_df = safe_cache(dbu_inc_df)

                merged_row_count = dbu_inc_df.count()

                validate_source_schema(
                    dbu_inc_df,
                    {"workspace_id": "string", "pipeline_id": "string",
                     "usage_date": "date", "compute_mode": "string",
                     "billing_origin_product": "string", "databricks_cost": "double"},
                    self.target_table, self.logger,
                )
                validate_no_negative_costs(
                    dbu_inc_df,
                    ["databricks_cost", "update_cost", "maintenance_cost"],
                    self.target_table, self.logger,
                )
                validate_currency_consistency(dbu_inc_df, "currency", self.target_table, self.logger)

                ensure_boolean_columns(self.target_table, ["workspace_covered"], logger=self.logger)
                target = DeltaTable.forName(spark, self.target_table)
                (target.alias("t")
                    .merge(
                        dbu_inc_df.alias("s"),
                        # cluster_id matched NULL-SAFE (<=>) - serverless rows
                        # carry NULL and a plain '=' would never match them,
                        # re-inserting ~96% of the spend on each overlapping run
                        # (§3.3/§5.4). The other key columns are non-nullable.
                        "t.workspace_id = s.workspace_id "
                        "AND t.pipeline_id = s.pipeline_id "
                        "AND t.usage_date = s.usage_date "
                        "AND t.cluster_id <=> s.cluster_id "
                        "AND t.billing_origin_product = s.billing_origin_product",
                    )
                    .whenMatchedUpdate(set={
                        "databricks_cost": "s.databricks_cost",
                        "update_cost": "s.update_cost",
                        "maintenance_cost": "s.maintenance_cost",
                        "compute_mode": "s.compute_mode",
                        "currency": "s.currency",
                        "sku_name": "s.sku_name",
                        "workspace_covered": "s.workspace_covered",
                        "updated_at": "current_timestamp()",
                    })
                    .whenNotMatchedInsert(values={
                        "workspace_id": "s.workspace_id",
                        "pipeline_id": "s.pipeline_id",
                        "usage_date": "s.usage_date",
                        "cluster_id": "s.cluster_id",
                        "billing_origin_product": "s.billing_origin_product",
                        "compute_mode": "s.compute_mode",
                        "databricks_cost": "s.databricks_cost",
                        "update_cost": "s.update_cost",
                        "maintenance_cost": "s.maintenance_cost",
                        "currency": "s.currency",
                        "sku_name": "s.sku_name",
                        "workspace_covered": "s.workspace_covered",
                        "created_at": "current_timestamp()",
                        "updated_at": "current_timestamp()",
                    })
                    .execute()
                )

                safe_unpersist(dbu_inc_df)
                get_merge_metrics(self.target_table, self.logger)

                validate_post_merge(
                    self.target_table, "usage_date",
                    start_dt, end_dt, merged_row_count, self.logger,
                )

            safe_unpersist(u)

            log_audit_run(self.audit_table, self.TABLE_NAME, start_dt, end_dt, "SUCCESS", merged_row_count, "")
            self.logger.info(
                f"Merged {merged_row_count} rows into {self.target_table} "
                f"for {start_dt} → {end_dt}."
            )

        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Run failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table, self.TABLE_NAME, start_dt, end_dt, "FAILED", 0, msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

In [ ]:
# =======================================================
# APP
# =======================================================
class PipelineDBUCostReporterApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        overlap_days = get_overlap_days(dbutils.widgets.get("overlap_days"), logger=logger)

        audit_table = build_table_fqn(catalog, schema, "dbspend360_audit_log")
        target_table = build_table_fqn(catalog, schema, "dbspend360_pipeline_dbu_cost")

        self.client = PipelineDBUCostClient(
            audit_table=audit_table,
            target_table=target_table,
            covered_table=build_table_fqn(catalog, schema, "dbspend360_covered_workspaces"),
            overlap_days=overlap_days,
            logger=logger,
        )

    def run(self):
        self.client.compute_and_merge_dbu_cost()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = PipelineDBUCostReporterApp()
app.run()